# <b>Exploratory Data Analysis</b>
## Setup: Imports And Loading Data

In [14]:
import pandas as pd

apc = pd.read_csv('../data/raw/apc.csv', sep = ';', encoding = 'latin-1')
matched_stops = pd.read_csv('../data/raw/matched_stops.csv', sep = ';', decimal=',', encoding = 'latin-1')
matched_trips = pd.read_csv('../data/raw/matched_trips.csv', sep = ';', decimal=',', encoding = 'latin-1')
thoreb = pd.read_csv('../data/raw/thoreb.csv', encoding = 'latin-1', low_memory = False)

print(f"apc:           {apc.shape}")
print(f"matched_stops: {matched_stops.shape}")
print(f"matched_trips: {matched_trips.shape}")
print(f"thoreb:        {thoreb.shape}")

apc:           (1598279, 27)
matched_stops: (56165, 25)
matched_trips: (1504, 20)
thoreb:        (150003, 13)


## Step 1 - Unique vehicles per file

In [11]:
print(f"apc:           {apc['vehicleCode'].nunique()}")
print(f"matched_stops: {matched_stops['vehicleNumber'].nunique()}")
print(f"matched_trips: {matched_trips['data.vehicles.vehicleNumber'].nunique()}")
print(f"thoreb:        {thoreb['vehicle'].nunique()}")

apc:           176
matched_stops: 23
matched_trips: 23
thoreb:        170


## Step 2 - Empty rows and duplicates

In [12]:
files = {
    'apc':           apc,
    'matched_stops': matched_stops,
    'matched_trips': matched_trips,
    'thoreb':        thoreb,
}

for name, df in files.items():
    empty_rows = df.isna().all(axis=1).sum()
    duplicates = df.duplicated().sum()
    print(f"{name}:")
    print(f"  empty rows: {empty_rows}")
    print(f"  duplicates: {duplicates}")

apc:
  empty rows: 0
  duplicates: 2
matched_stops:
  empty rows: 0
  duplicates: 0
matched_trips:
  empty rows: 0
  duplicates: 0
thoreb:
  empty rows: 1
  duplicates: 0


## Step 3 - GPS verification with Folium
Goal: plot stop coordinates on a map to verify they all fall within Halland.

In [15]:
import folium

# Get unique stops with coordinates from matched_stops
stops = matched_stops[['stopLabel', 'latitude', 'longitude']].drop_duplicates('stopLabel')
stops = stops.dropna(subset=['latitude', 'longitude'])

# Center map roughly on Halland
m = folium.Map(location=[56.7, 12.9], zoom_start=9)

for _, row in stops.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        popup=row['stopLabel'],
        color='blue',
        fill=True,
    ).add_to(m)

m.save('../data/results/map_matched_stops.html')
print(f"Plotted {len(stops)} unique stops")
print("Map saved to ../data/results/map_matched_stops.html")

Plotted 218 unique stops
Map saved to ../data/results/map_matched_stops.html


**Found:** "Fiktiv nod" entries with dummy coordinates (~0,0). 
Not a real stop, placeholder in Dilax system. To be filtered out in processing.

In [16]:
matched_stops[matched_stops['stopLabel'] == 'Fiktiv nod'][['stopLabel', 'latitude', 'longitude']].drop_duplicates()

,stopLabel,latitude,longitude
11080,Fiktiv nod,0.0,0.0


In [17]:
fiktiv = matched_stops[matched_stops['stopLabel'] == 'Fiktiv nod']
print(f"Rows with 'Fiktiv nod': {len(fiktiv)}")

Rows with 'Fiktiv nod': 2


**Note for later:** 
- Check overlap of the 23 vehicles in matched_stops vs matched_trips, same fleet?
- Check how many of those 23 also appear in apc and thoreb.
- Decide: filter to 23 vehicles only, or run two parallel analyses (full APC + 23-vehicle subset)?